# Investigate equations for rotation
For improvements on the notation or compactness of various variables

# Rotation matices
For investigations of better forms of various variables

In [1]:
import sympy
import timeit
import profile

In [5]:
phi, theta = sympy.symbols('phi theta', real=True)
alpha = sympy.symbols('alpha', real=True)

cp = sympy.cos(phi)
sp = sympy.sin(phi)
ct = sympy.cos(theta)
st = sympy.sin(theta)
Ry = lambda a: sympy.Matrix([[sympy.cos(a),0,sympy.sin(a)],[0,1,0],[-sympy.sin(a),0,sympy.cos(a)]])
Rz = lambda a: sympy.Matrix([[sympy.cos(a),-sympy.sin(a),0],[sympy.sin(a),sympy.cos(a),0],[0,0,1]])

Rz(phi) @ Ry(theta) @ Rz(alpha) @ Ry(-theta) @ Rz(-phi)

Matrix([
[((-sin(alpha)*sin(phi) + cos(alpha)*cos(phi)*cos(theta))*cos(theta) + sin(theta)**2*cos(phi))*cos(phi) - (-sin(alpha)*cos(phi)*cos(theta) - sin(phi)*cos(alpha))*sin(phi), ((-sin(alpha)*sin(phi) + cos(alpha)*cos(phi)*cos(theta))*cos(theta) + sin(theta)**2*cos(phi))*sin(phi) + (-sin(alpha)*cos(phi)*cos(theta) - sin(phi)*cos(alpha))*cos(phi), -(-sin(alpha)*sin(phi) + cos(alpha)*cos(phi)*cos(theta))*sin(theta) + sin(theta)*cos(phi)*cos(theta)],
[ ((sin(alpha)*cos(phi) + sin(phi)*cos(alpha)*cos(theta))*cos(theta) + sin(phi)*sin(theta)**2)*cos(phi) - (-sin(alpha)*sin(phi)*cos(theta) + cos(alpha)*cos(phi))*sin(phi),  ((sin(alpha)*cos(phi) + sin(phi)*cos(alpha)*cos(theta))*cos(theta) + sin(phi)*sin(theta)**2)*sin(phi) + (-sin(alpha)*sin(phi)*cos(theta) + cos(alpha)*cos(phi))*cos(phi),  -(sin(alpha)*cos(phi) + sin(phi)*cos(alpha)*cos(theta))*sin(theta) + sin(phi)*sin(theta)*cos(theta)],
[                                                                    (-sin(theta)*cos(alpha)*cos(th

In [3]:
# This is Rprime in SpinW or RtoZ in my functions

Rprime = Rz(phi) * Ry(theta) * Rz(-phi)
Rprime.simplify()
u = Rprime[:,0] + sympy.I*Rprime[:,1]
v = Rprime[:,2]

# by my simplifications
u_MS = sympy.Matrix([1,sympy.I,0]) + sympy.Matrix([cp*(ct-1), sp*(ct-1),st])*sympy.exp(sympy.I*phi)
# u_MS = sympy.Matrix([1,sympy.I,0]) + sympy.Matrix([cp*(ct-1), sp*(ct-1),st])*(sympy.cos(phi) + sympy.I*sympy.sin(phi))

hopefully_zero = u - u_MS
hopefully_zero = sympy.trigsimp(hopefully_zero)
hopefully_zero.simplify()
hopefully_zero  # Is actually zero, byt sympy struggles with Euler formula
u = u_MS


# Add exchange interactions

In [ ]:
Jxx, Jyy, Jzz, Jxy, Jxz, Jyz = sympy.symbols('Jxx, Jyy, Jzz, Jxy, Jxz, Jyz')
J = sympy.Matrix([[Jxx,Jxy,Jxz],[-Jxy,Jyy,Jyz],[-Jxz,-Jyz,Jzz]])
psi = sympy.symbols('psi')  # this will be the phase factor sits in exp(-i Qhkl.Nuvw)

A = u.T * J*sympy.exp(sympy.I*psi) * u.conjugate()
A.simplify()
A

In [ ]:
print(sympy.re(u))
print(sympy.im(u))

In [ ]:
Rprime

In [ ]:
import numpy as np
import scipy
M_SIZE = 4
test_matrices = np.random.random((100,M_SIZE,M_SIZE))+np.diag([10]*M_SIZE)

In [ ]:
N = 10_00
t_det = timeit.timeit("np.linalg.det(test_matrices)", number=N, globals=globals())
t_eig = timeit.timeit("np.linalg.eigvalsh(test_matrices)", number=N, globals=globals())
print(f'DET: {t_det}')
print(f'EIG: {t_eig}')

In [ ]:
N = 10_0000
t_det = timeit.timeit("np.linalg.cholesky(test_matrices[0])", number=N, globals=globals())
t_eig = timeit.timeit("scipy.linalg.cholesky(test_matrices[0])", number=N, globals=globals())
print(f'DET: {t_det}')
print(f'EIG: {t_eig}')

np.linalg.cholesky(test_matrices)[0]

In [1]:
from spinwaves import Crystal, MSG, Coupling, Atom, SpinW
import numpy as np
import timeit

def test_new_functionalities():
    ### SETUP
    P4mm = MSG.from_xyz_strings(generators=['x, y, z, +1'])
    atoms = [Atom(label='Cu', r=(0,0,0),   m=(0,1,0), s=1),] 
    crystal = Crystal(lattice_parameters=[3,8,8, 90,90,90], MSG=P4mm, atoms=atoms)

    magnetic_modulation = {
        'k':(1/2, 0, 0),
        'n':(1,0,0)
    }

    J = 1
    couplings = [Coupling(label='Ja', id1=0, id2=0, n_uvw=[1,0,0], J=J*np.eye(3,3))]

    ### CALCULATIONS
    sw = SpinW(crystal=crystal, magnetic_modulation=magnetic_modulation, couplings=couplings)

    return sw

sw = test_new_functionalities()

Qpath = np.linspace([0,0,0], [1,0,0], 5230)
# E, Sperp = sw.calculate_excitations(Qpath = Qpath)

N = 5
t_old = timeit.timeit("sw.determine_ES_new(Qhkl = Qpath)", number=N, globals=globals())
print(f'new: {t_old}')

N = 5
t_old = timeit.timeit("sw.calculate_excitations(Qpath = Qpath)", number=N, globals=globals())
print(f'old: {t_old}')



c:\users\stekiel\documents\github\spinwaves\spinwaves\spinw.py:472: RuntimeWarning: overflow encountered in expm1
  bose_factor = 1/np.expm1(np.abs(energies)/(0.08617333262*self.temperature)) \


ValueError: operands could not be broadcast together with remapped shapes [original->remapped]: (5230,2,2)->(5230,newaxis,newaxis) (5230,3,3,2,2)->(5230,3,3,newaxis,newaxis)  and requested shape (2,2)

In [20]:
# a = np.arange(8).reshape((2,2,2))
a = np.random.random((3,4,3,3))
b = np.random.random((3,4,5,3,3))

# c = np.block([[a,b],[b,a]])
# c = a[..., np.newaxis, np.newaxis] * b

a[0,0] = np.eye(3)
print(a[0,0])
print(b[0,0])

c = a @ b



print(c.shape)
c

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[[[0.82333767 0.82677401 0.14371828]
  [0.93580776 0.51846744 0.29567147]
  [0.9742262  0.78263872 0.83537154]]

 [[0.94290968 0.33575438 0.0011394 ]
  [0.03212964 0.81381205 0.73385857]
  [0.18161884 0.6494018  0.86662525]]

 [[0.26674472 0.93917753 0.4325058 ]
  [0.90961837 0.52985417 0.91531913]
  [0.53961954 0.84522265 0.50808637]]

 [[0.97088665 0.94929508 0.54639114]
  [0.30807917 0.20985624 0.52188219]
  [0.5262333  0.41988854 0.71673572]]

 [[0.46122078 0.86054487 0.97611633]
  [0.08668505 0.62457109 0.50313171]
  [0.19877416 0.66683072 0.57529917]]]


ValueError: operands could not be broadcast together with remapped shapes [original->remapped]: (3,4,3,3)->(3,4,newaxis,newaxis) (3,4,5,3,3)->(3,4,5,newaxis,newaxis)  and requested shape (3,3)

In [39]:
np.pi/6

0.5235987755982988